# 🟢 新手模式 | Beginner Mode

**不用写代码。** 拖动滑杆、点“运行实验”，看曲线和回放怎么变，再回答一个小问题，就完成一次探索。

**三步操作：**
1. 点菜单栏 Run ▶▶ Run All Cells（全部运行一次，等默认结果出来）；
2. 拖动下面的滑杆改一个关键参数；
3. 点 **▶ 运行实验**，看对比图和一句话解读。

**诚实说明：** 你看到的数字全部来自真实引擎（不是演示动画），和完整模式是同一口径。
新手模式只写 `labs/data/lite/` 下的完成记录，**不计入教师成绩册**；想拿实验成绩，请接着做对应的完整模式。

# A1 新手模式：加塞场景，什么时候刹车？


## 直觉问题

前车突然加塞到你前面，你什么时候踩刹车最合适？

- **太晚刹车**：撞上。
- **太早刹车**：没危险也急刹，后车追尾风险、乘客也不舒服。

本实验只让你调一个数：**TTC 阈值**——“预计几秒会撞上就刹车”。阈值越大，刹车越早。
（滑杆能真正起作用：新手版控制器把滑杆值作为触发阈值；基础版控制器里这个数是写死的 3.0，改配置不生效。）

**你的任务在最后**：找到“刚好不撞”的最小阈值。


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
sys.dont_write_bytecode = True

# 新手驱动：滑杆 -> 真实引擎 -> 对比图（引擎本身不在本本里，全部复用完整模式的同一批代码）
from common.lite_drivers import (
    plot_curves, plot_bars,
)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

LAST = {}  # 最近一次运行的结果（供“保存心得”格子使用）
print("✓ 新手驱动就绪：下面拖动滑杆、点运行即可")


In [ ]:
from common.lite_drivers import (
    run_a1_lite, a1_baseline, a1_replay, interpret_a1, save_lite_record, render_lite_metric_card,
)

ttc = widgets.FloatSlider(
    value=3.0, min=1.5, max=4.0, step=0.1,
    description="TTC 阈值 (s)：", style={"description_width": "110px"},
    layout=widgets.Layout(width="70%"),
)
run_btn = widgets.Button(description="▶ 运行实验", button_style="success")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output(wait=True)
        print(f"正在跑真 esmini 仿真（阈值 {ttc.value:.1f}s）……请稍等")
        base = a1_baseline()
        res = run_a1_lite(ttc.value)
        out.clear_output(wait=True)
        
        gb = [(t, g) for t, g in zip(base["times"], base["gaps"]) if g is not None]
        gs = [(t, g) for t, g in zip(res["times"], res["gaps"]) if g is not None]
        display(plot_curves(
            [(base["times"], base["speeds"], "基线 3.0s", "-"),
             (res["times"], res["speeds"], f"你的 {ttc.value:.1f}s", "-")],
            "时间 (s)", "自车车速 (m/s)", "自车车速响应：基线 vs 你的调参"))
        display(plot_curves(
            [([t for t, _ in gb], [g for _, g in gb], "基线 3.0s", "-"),
             ([t for t, _ in gs], [g for _, g in gs], f"你的 {ttc.value:.1f}s", "-")],
            "时间 (s)", "目标车相对间距 (m)", "跟驰相对间距：基线 vs 你的调参"))
        display(a1_replay(res))
        
        badge_text = "🔴 发生追尾碰撞" if res["collision"] else "🟢 零碰撞安全刹停"
        badge_color = "red" if res["collision"] else "green"
        metrics_list = [
            ("最小 TTC", f"{res['min_ttc']} s" if res['min_ttc'] else "--"),
            ("最小间距", f"{res['min_gap']} m"),
            ("刹车介入", f"{res['brake_steps']} 步"),
            ("碰撞状态", "追尾碰撞" if res["collision"] else "安全避让"),
        ]
        display(render_lite_metric_card("A1 加塞避碰", badge_text, badge_color, metrics_list, interpret_a1(res)))
        
        save_lite_record("A1", {"ttc_threshold": ttc.value},
                         {"min_ttc": res["min_ttc"], "min_gap": res["min_gap"],
                          "collision": res["collision"]})
        LAST.clear()
        LAST.update({"exp": "A1", "params": {"ttc_threshold": ttc.value},
                     "metrics": {"min_ttc": res["min_ttc"],
                                 "min_gap": res["min_gap"],
                                 "collision": res["collision"]}})

run_btn.on_click(on_run)
display(ttc, run_btn, out)
on_run(None)

## 🎯 找区间小任务

拖动滑杆，回答下面两个问题（写在下一个格子的心得里）：

1. 阈值从 2.5 调小到 2.0，发生了什么变化？（提示：2.5 不撞、2.0 撞上——“刚好不撞”的阈值就在这之间，步长 0.1 可以精确定位。）
2. “刚好不撞”的最小阈值是多少？（答案约 2.1s）再把它调大 0.5，最小间距增加了多少——
多出来的这段间距，就是“提前刹车买来的安全余量”，你觉得值不值？

说明：本实验把“间距 <0.5m”记为碰撞（A1 基础版的判据），所以最小间距不会出现负数——看“碰撞”那一栏有没有变 True。


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from common.lite_drivers import save_lite_record

note = widgets.Textarea(
    value="",
    placeholder="写一句话：你找到了什么规律？（例如：阈值越大，刹车越早，但……）",
    description="我的心得：",
    layout=widgets.Layout(width="100%", height="80px"),
)
save_btn = widgets.Button(description="💾 保存心得与打卡记录", button_style="info")
save_out = widgets.Output()

def _on_save(b):
    with save_out:
        save_out.clear_output(wait=True)
        if not LAST:
            print("⚠️ 请先运行一次实验（上一个格子的“运行实验”按钮），再保存。")
            return
        ans = note.value.strip()
        if not ans:
            print("⚠️ 请在上方输入框写下一句寻优心得后再打卡保存！")
            return
        path = save_lite_record(LAST["exp"], LAST["params"], LAST["metrics"], note=ans)
        display(HTML("<span style='color: #16a34a; font-weight: bold;'>🎉 恭喜！新手探索打卡成功，已同步归档入预习数据库！</span><br/><span style='color: #64748b; font-size: 12px;'>注：新手模式完成记录独立于 29+1 课程成绩册，不影响期末考核。</span>"))

save_btn.on_click(_on_save)
display(note, save_btn, save_out)

---

## 下一步 | Next

- 完成上面的“找区间”小任务后，把你的心得保存下来（上一个格子）。
- 想拿实验成绩、学完整算法：去做对应的完整模式（同目录下不带 `_lite` 的 notebook）。
- 新手模式的意义是先建立直觉：参数变了、结果往哪个方向变、变化有多大。

完整模式：`A1_cutin_aeb_pro.ipynb`
